# Visualization

Written by:

* Philip Ilten (University of Cincinnati)

This notebook provides a short guide on how to use the visualization tool VISTAS, developed by the [MLhad](https://uchep.gitlab.io/mlhad-docs/) group for Pythia, a general purpose Monte Carlo event generator for high energy particle physics.

## Requirements

Before running this notebook, we need to set up our environment. First, we install and import the `wurlitzer` module. This allows programs that have C-like backends to write their output to the Python console. In short, this allows the output of Pythia to be displayed in this notebook.

In [ ]:
# Redirect the C output of Pythia to the notebook.
!pip install wurlitzer
from wurlitzer import sys_pipes_forever

sys_pipes_forever()

Next, we need to install the Pythia module.

In [ ]:
# Install and import the Pythia module.
!pip install pythia8mc
import pythia8mc as pythia8

Finally, we need to download and import the visualization module `vistas`.

In [ ]:
# Download the visualization module.
!wget -q -N https://gitlab.com/mcgen-ct/tutorials/-/raw/main/.full/pythia/vistas.py

# Import `vistas` module.
import vistas

We also define a few top-level variables that may need to be modified if you are having issues with the visualization. You should hopefully not need to change these, but can if you run into problems.

In [ ]:
# Sometimes Phoenix loads the detector geometry slowly, and so the
# display configuration does not load correctly. If this happens,
# you can switch the Phoenix display URL from 'trackml' to
# 'playground'. But, when using the playground, the Phoenix Menu
# is no longer available, so toggling parts of the event or applying
# cuts is not possible.
playground = "https://hepsoftwarefoundation.org/phoenix/playground"
trackml = "https://hepsoftwarefoundation.org/phoenix/trackml"
phoenix = trackml

# If a web browser window is not opening, you can set this to a higher
# value (in seconds) and then click the printed link.
sleep = 5

## Introduction

The idea behind VISTAS, a Visualization Interface for Particle Collision Simulations, is to provide a more intuitive way to visualize high energy particle collisions as produced by Monte Carlo (MC) event generators. In this initial version of VISTAS, the focus has been to convert the output of Pythia into an interactive visual representation of the event generation process, where the different steps of high energy physics MC event generation are shown, including hard process, parton shower, and hadronization. This is different from experimental event displays, where events are displayed as reconstructed by the detector from real data.

VISTAS uses the [Phoenix event display framework](https://github.com/HSF/phoenix), which is focused on experimental event displays. The output of Pythia events is parsed into a format which can be used in this framework and then displayed interactively.

## The event record

Let us start with a relatively clean first event, where we generate the process $e^+ e^- \to Z^0 \to \mu^+ \mu^-$ with Pythia, something that might be observed at the Large Electron-Positron collider (LEP).

In [ ]:
# Create the Pythia object.
pythia = pythia8.Pythia("", False)
pythia.readString("Print:quiet = on")

# Set beams to electron (11) and positron (-11).
pythia.readString("Beams:idA = 11")
pythia.readString("Beams:idB = -11")

# Set center-of-mass energy at the Z pole.
pythia.readString("Beams:eCM = 91.2")

# Enable Z production.
pythia.readString("WeakSingleBoson:ffbar2gmZ = on")

# Turn off all Z decays except to muons.
pythia.readString("23:onMode = off")
pythia.readString("23:onIfMatch = 13 -13")

# Turn off some parts of the generation. We do this to simplfy the event, but
# will turn them back on later.
pythia.readString("PDF:lepton = off")
pythia.readString("PartonLevel:ISR = off")
pythia.readString("PartonLevel:FSR = off")

# Initialize Pythia.
pythia.init()

We can then generate an event and list the event using the standard Pythia output.

In [ ]:
# Generate an event.
pythia.next()

# List the Pythia event.
pythia.event.list()

While this type of event record is specific to Pythia, many MC generators have similar internal formats which are a list of particles with information on the connections between those particles. Here, each row of the table corresponds to a particle and each column information for that specific particle. These columns are as follows.

- `no`: the index number of the particle in the Pythia event record.
- `id`: the PDG particle identity code of the particle. A complete specification of the PDG codes is found in the [Review of Particle Physics](https://pdg.lbl.gov/). An online listing is available from the [PDG MC numbering review](http://pdg.lbl.gov/2022/reviews/rpp2022-rev-monte-carlo-numbering.pdf).
- `name`: the particle name, within brackets for initial or intermediate particles and without for final-state ones.
- `status`: the part of the event generation that produced this particle.
- `mothers`: in general, the range of indices of the mother particles that produced this particle, but there are some subtleties about how to exactly interpret these numbers.
- `daughters`: in general, the range of indices of the daughter particles that this particle produced, but again there are some subtleties about how to exactly interpret these numbers.
- `colours`: the colour flowing through this particle, both color and anti-color..
- `p_x`, `p_y`, `p_z`, and `e`: the components of the momentum four-vector ($p_x$ , $p_y$ , $p_z$, $E$), in units of GeV with $c = 1$.
- `m`: the mass in GeV/$c^2$.

Here, we can see for example that a `Z0` with index `5` has daughters `8` and `9` which are a `mu-` and `mu+`, respectively. This `Z0` comes from mothers `3` and `4` which correspond to the `e-` and `e+` from the beam. You will notice that there can be multiple copies of what appears to be the same particle in the event; we will discuss what these mean later once we have a better idea of how an event is generated.

## Visualizing the event

We can think of this type of event record as a graph, where the vertices of the graph correspond to the production or end points of each particle, and the links are the particles themselves. This picture of the event as a graph leads to a natural way of visualizing the event. In three dimensions, we can plot this graph structure, where every line corresponds to a particle, and the direction of the line corresponds to the direction of its momentum. It is important to remember that this type of visualization **does not** correpond to the space-time representation of this event. We can use the VISTAS tool to perform this visualization.

In [ ]:
# Create the VISTAS veiwer. The Pythia object is passed as an argument.
viewer = vistas.Vistas(pythia)

# Display the event.
viewer.display(sleep=sleep, phoenix=phoenix)

You should see what looks like a light purple skewed "X". Sometimes, a detector incorrectly displays. This is an issue with Phoenix, where the detector geometry finishes loading after the configuration file. If this occurs, exit the viewer page and just try running again. If the problem persists, you can try changing the Phoenix viewer by setting `phoenix = playground` in the final cell of the introduction. Hopefully you should be able to see the event, but the "Phoenix Menu" on the right-hand side will no longer display.

If the cell above does not automatically open a new webpage with the visualization, then click the link that is printed before the cell completes its execution. If you need more time to click the link, pass the argument then you can change the value of `sleep` in the final cell of the introduction. This is the number of seconds the link is active. If the page correctly opens, but no event is displayed, try rerunning the cell, and optionally try a slightly longer `sleep` time.

You can inspect details for each particle, the purple lines here, by clicking the icon that looks like a cursor pointing to a small circle. For each particle the following details are given, with the corresponding relevant `Particle` function call in Pythia. All observables are given in the lab frame, which is not necessarily the visualization frame, more details on that later.

1. *category*: the part of the Pythia generation that produced this particle.
2. *name (PDG ID)*: both the particle name and Particle Data Group ID (`name`, `id`).
3. *status*: Pythia status and its corresponding meaning (`status`).
4. *px, py, pz, E*: momentum four-vector (`p`).
5. *pT*: transverse momentum (`pT`).
6. *charge x 3*: electromagnetic charge, multiplied by a factor of 3 (`chargeType`).
7. *color index*: color and anti-color of the particle (`col`, `acol`).
8. *index*: Pythia event record index (`index`).

Try exploring the information from particles, and connecting these back with the Pythia event, either through the event listing above or directly interacting with the `pythia.event` object. The colors here also have meaning, the darker the color, the higher the energy (in the lab frame) of that particle.

In [ ]:
# Check the momentum four-vector of the Z0.
print(pythia.event[5].p())

On the right, there is a menu labeled *Phoenix Menu*. There should be a row labeled *Event Data* with *Tracks* underneath. Within these tracks, categories of the particles from the event generation are shown. Here, we only have the hard process so only the *hard process* category should display. You can apply cuts on the lab frame $\phi$, $\eta$, and $p_T$ for each particle by clicking the arrow button next to each category and then the gear next to *Cut Options*. Note, these cuts only apply to that specific category. Try removing particles with high pseudorapidity.

 If we click the $\eta$ `-4` lower cut, we lose the `e+` beam and if we click the $\eta$ `+4` upper cut, we lose the `e-` beam and the `Z0`.


So far, this visualization is rather underwhelming, so let us move onto more complex (and realistic) examples!

## Event generation

Most MC generators use a factorized approach to produce an event. In a generator like Pythia, a physics scale, such as particle energy or transverse momentum, defines this factorization. The MC generator starts at the hardest (highest) scale of the event and evolves the particles in the event to softer (lower) scales. In Pythia (and other event generators) we oftentimes have the following categories of particle production in the event, ranked from hard to soft. Depending on the literature, you may see these grouped or ordered differently. For example, in Pythia the *MPI*, *ISR*, *FSR*, and *beam* are all evolved to softer scales together, so there is not a given ordering for each category.

1. *hard process*: this is the hardest sub-process of the event and and is calculated using perturbative quantum field theory (QFT). In our LEP example, this corresponds to the $e^+ e^- \to Z^0 \to \mu^+ \mu^-$ process that we already explored.
2. *MPI*: multi-parton interactions, or additional sub-processes from partons within the colliding beams. In our LEP example, we do not have MPI since the beams are not composite particles. However, if we used proton beams we could have additional quarks or gluons colliding.
3. *ISR* (parton shower): initial state radiation from the incoming particles to the hard process or MPI. The emissions of this radiation is calculated using perturbative QFT.
4. *FSR* (parton shower): final state radiation from any of the outgoing particles. Again these emissions are calcualted using perturbative QFT.
5. *parton prep*: quarks and gluons that need to be reshuffled in preparation for hadronization. This is a more Pythia-specific category.
6. *hadronization*: the quarks and gluons produced from harder scales in the event are combined together into bound hadrons.
7. *decay*: unstable particles decays.
8. *beam*: beam particles and remnants.
9. *color flow*: lines that connect color singlets in the event.

In our LEP example, we only have the *hard process* category so far, but we can change the Pythia generation so that some of these other categories begin to show up. Let us first add *ISR*.

In [ ]:
# Turn on ISR. We also need to treat the electron as having a PDF.
pythia.readString("PDF:lepton = on")
pythia.readString("PartonLevel:ISR = on")

# Reinitialize Pythia.
pythia.init()

# Generate events. Since ISR on the electrons does not always occur, we may
# need  to generate a few events.
isr = False
for idx in range(100):
    pythia.next()
    for prt in pythia.event:
        if 41 <= prt.statusAbs() <= 49:
            isr = True
            break
    if isr:
        break

# List the event.
pythia.event.list()

Here, we see that we have two photons with a status code `63` which are the *beam remnants* and one or more photons with status code `43` that are coming from the *ISR*. Specifically, the hard process electron/positron dipole radiates a photon. We can visualize this event with the following.

In [ ]:
# Display the event.
viewer.display(sleep=sleep, phoenix=phoenix)

Now there are a few more categories in the *Phoenix Menu*. Try toggling on and off some of these categories to see how the *ISR*, *beams*, and *beam remnants* are shown in the event display. Let us add FSR.

In [ ]:
# Turn on FSR.
pythia.readString("PartonLevel:FSR = on")

# Reinitialize Pythia.
pythia.init()

# Generate events. Since FSR on the muon does not always occur, we may
# need  to generate a few events.
fsr = False
for idx in range(100):
    pythia.next()
    for prt in pythia.event:
        if 51 <= prt.statusAbs() <= 59:
            fsr = True
            break
    if fsr:
        break

# List the event.
pythia.event.list()

The event record is even longer now, with at least one photon with status `51`. This corresponds to the muon/anti-muon dipole radiating a photon. Let us visualize this.

In [ ]:
# Display the event.
viewer.display(sleep=sleep, phoenix=phoenix)

So far, we have not seen decays since neither the electron nor muon decay. We can switch the final state muon pair to a tau pair, where the tau will decay.

In [ ]:
# Turn off all Z decays except to muons.
pythia.readString("23:onMode = off")
pythia.readString("23:onIfMatch = 15 -15")

# Reinitialize Pythia.
pythia.init()

# Generate an event.
pythia.next()

# List the event.
pythia.event.list()

We display the event and now see decay products of the tau pair.

In [ ]:
# Display the event.
viewer.display(sleep=sleep, phoenix=phoenix)

Up to this point, we have not yet had any perturbative QCD so we have not seen any hadronization. To get hadronization we need quarks and gluons, so we can change decay of the $Z^0$ to a $d/\bar{d}$ pair.

In [ ]:
# Turn off all Z decays down quarks.
pythia.readString("23:onMode = off")
pythia.readString("23:onIfMatch = 1 -1")

# Reinitialize Pythia.
pythia.init()

# Generate an event.
pythia.next()

# List the event.
pythia.event.list()

As we might expect, this results in a significantly busier event. Let us see what this looks like.

In [ ]:
# Display the event.
viewer.display(sleep=sleep, phoenix=phoenix)

## Beyond LEP

We have been exploring LEP events up to this point which has allowed us to explore most of the aspects of MC event generation, except *MPI*. Before we move on to and *MPI* example, let us first consider swapping out the positron beam with a proton beam to give us an Electron Ion Collider (EIC) scenario. We can set the beam energies at the high end of the EIC energy range.

In [ ]:
# Create the Pythia object.
pythia = pythia8.Pythia("", False)
pythia.readString("Print:quiet = on")

# Set beams to electron (11) and proton (2212).
pythia.readString("Beams:idA = 11")
pythia.readString("Beams:idB = 2212")

# Set asymmetric beam energies (in GeV).
pythia.readString("Beams:frameType = 2")
pythia.readString("Beams:eA = 18")
pythia.readString("Beams:eB = 275")

# Enable DIS.
pythia.readString("WeakBosonExchange:ff2ff(t:gmZ) = on")

# Preserve DIS kinematics.
pythia.readString("SpaceShower:dipoleRecoil = on")

# Initialize Pythia.
pythia.init()

We can now generate an event, and visualize. Since this is $t$-channel exchange, there is no record of the $\gamma^*/Z^0$ (or hadronic vector resonance) in the event record.

In [ ]:
# Generate the event.
pythia.next()

In [ ]:
# Visualize.
viewer = vistas.Vistas(pythia)
viewer.display(sleep=sleep, phoenix=phoenix)

Unlike the LEP scenario, the lab frame is not the same as the center-of-mass frame of the hard process. With VISTAS, we can use the `opts` dictionary to configure the visualization options. The `frame` option allows us to specify visualizing the event in the `beam` frame, the `hard process` frame, or the `lab` frame. By default the `hard process` frame is used. By changing this, we can look at the event in the lab frame.

In [ ]:
# View the event in the lab frame (no boost is performed).
viewer.opts["frame"] = "lab"
viewer.display(sleep=sleep, phoenix=phoenix)

It is also possible to boost to whatever frame you might like, by boosting the Pythia event itself before calling the visualization.

We have not yet seen any *MPI* yet, as both beams need to be hadrons. For this, we can run a Large Hadron Collider (LHC) configuration.

In [ ]:
# Create the Pythia object.
pythia = pythia8.Pythia("", False)
pythia.readString("Print:quiet = on")

# Set beams to proton (2212) and proton (2212).
pythia.readString("Beams:idA = 2212")
pythia.readString("Beams:idB = 2212")

# Set asymmetric beam energies (in GeV).
pythia.readString("Beams:frameType = 2")
pythia.readString("Beams:eA = 6500")
pythia.readString("Beams:eB = 6500")

# Enable top pair production.
pythia.readString("Top:gg2ttbar = on")

# Initialize Pythia.
pythia.init()

In [ ]:
# Generate the event.
pythia.next()

In [ ]:
# Visualize.
viewer = vistas.Vistas(pythia)
viewer.display(sleep=sleep, phoenix=phoenix)

Here, we have produced a $t\bar{t}$ and there are a number of *MPI* in the event. Looking at this event, notice that the MPI are almost all parallel to the beam direction, but that the $W$ decays from the $t\bar{t}$ system are not. Why is this?

We can also try different types of LHC events, such as Higgs production.

In [ ]:
# Create the Pythia object.
pythia = pythia8.Pythia("", False)
pythia.readString("Print:quiet = on")

# Set beams to proton (2212) and proton (2212).
pythia.readString("Beams:idA = 2212")
pythia.readString("Beams:idB = 2212")

# Set the center-of0 (in GeV).
pythia.readString("Beams:frameType = 2")
pythia.readString("Beams:eA = 6500")
pythia.readString("Beams:eB = 6500")

# Enable Higgs VBF production.
pythia.readString("HiggsSM:ff2Hff(t:ZZ) = on")

# Turn off all Higgs decays except to photons.
pythia.readString("25:onMode = off")
pythia.readString("25:onIfMatch = 22 22")

# Initialize Pythia.
pythia.init()

In [ ]:
# Generate the event.
pythia.next()

In [ ]:
# Visualize.
viewer = vistas.Vistas(pythia)
viewer.display(sleep=sleep, phoenix=phoenix)

Finally, we can even visualize heavy ion events. Here, we need to be a little careful, because the Phoenix viewer can only handle so many particles and will crash if the event multiplicity is to high. Note, initializing Pythia in this mode takes some time, so be patient.

In [ ]:
# Create the Pythia object.
pythia = pythia8.Pythia("", False)
pythia.readString("Print:quiet = on")

# Set beams to Pb (1000822080) Pb (1000822080).
pythia.readString("Beams:idA = 1000822080")
pythia.readString("Beams:idB = 1000822080")

# Provide fit values for speed purposes (these lines can be omitted).
pythia.readString("HeavyIon:SigFitNGen = 0")
pythia.readString("HeavyIon:SigFitDefAvNDb = 1.13")
pythia.readString("HeavyIon:SigFitDefPar = 1.57,13.35,0.18")

# Set center-of-mass energy.
pythia.readString("Beams:eCM = 5020")

# Enable soft QCD production.
pythia.readString("SoftQCD:all = on")

# Initialize Pythia.
pythia.init()

In [ ]:
# Generate an event, but not too large of an event.
while pythia.event.size() < 5000 or pythia.event.size() > 10000:
    pythia.next()

When visualizing the event, there will be a warning about the number of scatters present. This is expeceted, given how Pythia and Angantyr count MPI. There also may be warnings about missing colors/anti-colors, which indicate that some color lines may be missing.

In [ ]:
# Visualize.
viewer = vistas.Vistas(pythia)
viewer.display(sleep=sleep, phoenix=phoenix)

## Visualization Options

There are a number of visualization options available in VISTAS that can be controlled by the `opts` dictionary of a `Vistas` object. Use the built in Python `help` to find these options.

In [ ]:
help(viewer.options)

With the EIC configuration we already tested the `frame` option. With this heavy ion output, we can test the `mpi` option, where we can place all sub-collisions on top of each other.

In [ ]:
viewer.opts["mpi"] = 0
viewer.display(sleep=sleep, phoenix=phoenix)

It can be easy to modify the options to the point where the visualization is unusable. To recover to the default settings, the `options` method can be used.

In [ ]:
viewer.opts = viewer.options()

In some cases it can be useful to toggle parts of the generation off in the generation. This is accomplished with the `show` key. Try and visualize the event where only the hard process and MPI are toggled on. First, let us work with a slightly smaller event again.

In [ ]:
# Create the Pythia object.
pythia = pythia8.Pythia("", False)
pythia.readString("Print:quiet = on")

# Set beams to proton (2212) and proton (2212).
pythia.readString("Beams:idA = 2212")
pythia.readString("Beams:idB = 2212")

# Set the center-of0 (in GeV).
pythia.readString("Beams:frameType = 2")
pythia.readString("Beams:eA = 6500")
pythia.readString("Beams:eB = 6500")

# Enable top pair production.
pythia.readString("HiggsSM:ff2Hff(t:ZZ) = on")

# Initialize Pythia.
pythia.init()

In [ ]:
# Generate the event.
pythia.next()

In [ ]:
# Visualize.
viewer = vistas.Vistas(pythia)
viewer.display(sleep=sleep, phoenix=phoenix)

In [ ]:
# Only show the hard process and MPI.
viewer.opts["show"] = ["hard process", "MPI"]
# Visualize.
viewer.display(sleep=sleep, phoenix=phoenix)

Jets are groups of particles clustered together using a specific algorithm. The `jets` option allows for jets to be built and then viewed. Keep in mind that the jets are always built in the initial unboosted frame of the event, but then boosted to visualization frame. Try building jets with the anti-$k_T$ algorithm.

In [ ]:
# Reset the options.
viewer.opts = viewer.options()
# Turn on anti-kT jets.
viewer.opts["jets"]["algorithm"] = "akt"
# Visualize.
viewer.display(sleep=sleep, phoenix=phoenix)

It is also possible to change the visual style of the event. For example, the length of the displayed line for each particle is by default the same for all particles. It is possible to make this length proportional to observables of that particle (in the lab frame). Try visualizing the event where the length of the particle lines are proportional to energy on a linear scale.

In [ ]:
# Reset the options.
viewer.opts = viewer.options()
# Change length to be proportional to energy.
viewer.opts["length"]["scale"] = "log"
viewer.opts["length"]["observable"] = "p.e()"
# Change the log skew to linear.
viewer.opts["length"]["skew"] = 1
# Visualize.
viewer.display(sleep=sleep, phoenix=phoenix)

Try the same, but now with the default skew. Also change, the color to be constant and set such that the maximum color is `0`.

In [ ]:
# Reset the options.
viewer.opts = viewer.options()
# Change the color to be constant.
viewer.opts["color"]["scale"] = "constant"
viewer.opts["color"]["max"] = 0
# Change length to be proportional to energy.
viewer.opts["length"]["scale"] = "log"
viewer.opts["length"]["observable"] = "p.e()"
# Visualize.
viewer.display(sleep=sleep, phoenix=phoenix)